# Exploring the streamlake gold tables

Run `streamlake demo` (or the Kafka path) first, then execute the cells below.
The notebook is read-only: it opens `warehouse/streamlake.duckdb` in DuckDB's
read-only mode so it can run while you inspect the lake.

In [ ]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import pandas as pd

pd.options.display.float_format = "{:,.2f}".format

warehouse = Path("warehouse") / "streamlake.duckdb"
if not warehouse.exists():
    warehouse = Path("..") / "warehouse" / "streamlake.duckdb"
if not warehouse.exists():
    raise FileNotFoundError(
        f"{warehouse} not found - run `streamlake demo` from the repo root first"
    )

con = duckdb.connect(str(warehouse), read_only=True)
con.execute("SELECT * FROM analytics.ops_model_runs ORDER BY run_at DESC LIMIT 6").df()

In [ ]:
revenue = con.execute(
    """
    SELECT order_date, orders, paid_orders, net_revenue, avg_order_value
    FROM analytics.gold_daily_revenue
    ORDER BY order_date
    """
).df()
revenue.tail(10)

In [ ]:
funnel = con.execute(
    """
    SELECT step, step_order, SUM(sessions) AS sessions
    FROM analytics.gold_funnel
    GROUP BY step, step_order
    ORDER BY step_order
    """
).df()

funnel["pct_of_sessions"] = (
    funnel["sessions"] / funnel.loc[funnel["step"] == "page_view", "sessions"].iloc[0] * 100
).round(2)
funnel

In [ ]:
products = con.execute(
    """
    SELECT product_id, add_to_cart_events, units, gross_revenue
    FROM analytics.gold_product_performance
    ORDER BY gross_revenue DESC
    LIMIT 10
    """
).df()

ax = products.set_index("product_id")["gross_revenue"].plot.bar(
    figsize=(9, 4), color="#2f6f4e", title="Top products by paid revenue"
)
ax.set_xlabel("product")
ax.set_ylabel("gross revenue (USD)")
plt.tight_layout()
plt.show()
products

In [ ]:
con.close()

## Notes

- Gold tables are rebuilt from bronze on every `streamlake transform`; the
  notebook never mutates them.
- `ops_model_runs` is the quickest way to see whether the latest transform
  succeeded and how expensive each model was.
- For a larger view, change the revenue query to aggregate by week with
  `date_trunc('week', order_date)`.